In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

In [2]:
val packageVersion = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % packageVersion)  // use programmatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

packageVersion: String = "0.0.1"

In [3]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan

import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._

import org.dataprov.dp.wringlet.ProvenanceApi._
import org.dataprov.dp.wringlet.LogicalPlanWithProvenance
import org.dataprov.dp.wringlet.SparkProvenanceExtension
import org.dataprov.dp.wringlet.SemiWhyProvenanceBuilder
import org.dataprov.dp.wringlet.DisplayStringProvenanceBuilder
import org.dataprov.dp.wringlet.ProvenanceBuilder
import org.dataprov.dp.wringlet.BooleanProvenanceBuilder
import org.dataprov.dp.wringlet.FullWhyProvenanceBuilder


import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._
import org.dataprov.dp.wringlet.ProvenanceApi._
import org.dataprov.dp.wringlet.LogicalPlanWithProvenance
import org.dataprov.dp.wringlet.SparkProvenanceExtension
import org.dataprov.dp.wringlet.SemiWhyProvenanceBuilder
import org.dataprov.dp.wringlet.DisplayStringProvenanceBuilder
import org.dataprov.dp.wringlet.ProvenanceBuilder
import org.dataprov.dp.wringlet.BooleanProvenanceBuilder
import org.dataprov.dp.wringlet.FullWhyProvenanceBuilder

# Predefined provenance operators 

In [4]:
val spark = SparkSession.builder()
  .master("local[*]")
  .appName("notebook-ctable-boolean-provenance")
  .withExtensions(
    new SparkProvenanceExtension(
      provenanceBuilder = SemiWhyProvenanceBuilder
    )
  )
  // .config("spark.jars", s"../target/scala-2.13/dp-spark_2.13-$packageVersion.jar")
  // .config("spark.sql.extensions", "org.dataprov.dp.wringlet.SparkProvenanceExtension")
  .config("spark.provenance.enabled", "true")
  .getOrCreate()

println(s"Spark provenance enabled: ${spark.conf.get("spark.provenance.enabled")}")

// Set log level to ERROR to reduce verbosity
spark.sparkContext.setLogLevel("ERROR")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/10 15:20:35 INFO SparkContext: Running Spark version 4.1.1
26/07/10 15:20:35 INFO SparkContext: OS info Mac OS X, 26.4.1, aarch64
26/07/10 15:20:35 INFO SparkContext: Java version 17.0.10+7
26/07/10 15:20:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/10 15:20:35 INFO ResourceUtils: ==============================================================
26/07/10 15:20:35 INFO ResourceUtils: No custom resources configured for spark.driver.
26/07/10 15:20:35 INFO ResourceUtils: ==============================================================
26/07/10 15:20:35 INFO SparkContext: Submitted application: notebook-ctable-boolean-provenance
26/07/10 15:20:35 INFO SecurityManager: Changing view acls to: mac-ABALLA16
26/07/10 15:20:35 INFO SecurityManager: Changing modify acls to: mac-ABALLA16
26/07/10 15:20:35 INFO SecurityManager: Ch

Spark provenance enabled: true


spark: SparkSession = org.apache.spark.sql.classic.SparkSession@6ab1b8c3

In [5]:
val df0: DataFrame = spark.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("d", "b", "e"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C")

df0.show()

val dfWithProvenance = df0.addProvenanceColumn(col("A"))
dfWithProvenance.printSchema()
dfWithProvenance.show(false)

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+

root
 |-- A: string (nullable = true)
 |-- B: string (nullable = true)
 |-- C: string (nullable = true)
 |-- _provenance_tag: string (nullable = true)

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |a              |
|d  |b  |e  |d              |
|f  |g  |e  |f              |
+---+---+---+---------------+



df0: DataFrame = [A: string, B: string ... 1 more field]
dfWithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [6]:
val df2WithProvenance : DataFrame = dfWithProvenance
    .select("A", "B")
    .join(dfWithProvenance.select("B", "C"), "B")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df2WithProvenance.show(false)



+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |[a]            |
|a  |b  |e  |[a, d]         |
|d  |b  |c  |[d, a]         |
|d  |b  |e  |[d]            |
|f  |g  |e  |[f]            |
+---+---+---+---------------+



df2WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [7]:
dfWithProvenance.createOrReplaceTempView("df_with_prov")
val df3WithProvenance: DataFrame = spark.sql("""
    SELECT A, B, t1.C
    FROM (
        SELECT A, C
        FROM df_with_prov
    ) AS t1
    JOIN (
        SELECT B, C
        FROM df_with_prov
    ) AS t2
    ON t1.C = t2.C
    ORDER BY A, B, t1.C
""")

df3WithProvenance.show(false)

val df4WithProvenance = df2WithProvenance.union(df3WithProvenance).distinct().orderBy("A", "B", "C")
df4WithProvenance.show(false)

val df5WithProvenance = df4WithProvenance.select("A", "C").distinct().orderBy("A", "C")
df5WithProvenance.show(false)

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |[a]            |
|d  |b  |e  |[d]            |
|d  |g  |e  |[d, f]         |
|f  |b  |e  |[f, d]         |
|f  |g  |e  |[f]            |
+---+---+---+---------------+

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |[a]            |
|a  |b  |e  |[a, d]         |
|d  |b  |c  |[d, a]         |
|d  |b  |e  |[d]            |
|d  |g  |e  |[d, f]         |
|f  |b  |e  |[f, d]         |
|f  |g  |e  |[f]            |
+---+---+---+---------------+

+---+---+---------------+
|A  |C  |_provenance_tag|
+---+---+---------------+
|a  |c  |[a]            |
|a  |e  |[a, d]         |
|d  |c  |[d, a]         |
|d  |e  |[d]            |
|f  |e  |[f]            |
+---+---+---------------+



df3WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]
df4WithProvenance: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 2 more fields]
df5WithProvenance: Dataset[org.apache.spark.sql.Row] = [A: string, C: string ... 1 more field]

In [8]:
val df6WithProvenance = dfWithProvenance.groupBy("C").agg(collect_list("A").as("A_list")).orderBy("C")
df6WithProvenance.show(false)   

+---+------+---------------+
|C  |A_list|_provenance_tag|
+---+------+---------------+
|c  |[a]   |[a]            |
|e  |[d, f]|[d, f]         |
+---+------+---------------+



df6WithProvenance: Dataset[org.apache.spark.sql.Row] = [C: string, A_list: array<string> ... 1 more field]

# Custom provenance operators

In [9]:
import org.apache.spark.sql.types.{DataType, StringType}

// Example of a custom string-based provenance builder
object BracketStringProvenanceBuilder extends ProvenanceBuilder {
  val joinOperator = "&"
  val aggregateOperator = " || "
  val distinctOperator = " + "

  override val provType: DataType = StringType

  override def single(attr: Attribute): Expression = {
    Concat(Seq(Literal("["), Cast(attr, StringType), Literal("]")))
  }

  override def join(
      left: Attribute,
      right: Attribute,
  ): Expression = {
    val leftCast = Cast(left, StringType)
    val rightCast = Cast(right, StringType)

    val matchedTag = If(
      And(IsNotNull(left), IsNotNull(right)),
      Concat(Seq(leftCast, Literal(joinOperator), rightCast)),
      Cast(Literal(null), StringType)
    )

    val leftOnly = If(IsNotNull(left), leftCast, Cast(Literal(null), StringType))
    val rightOnly = If(IsNotNull(right), rightCast, Cast(Literal(null), StringType))

    Coalesce(Seq(matchedTag, leftOnly, rightOnly))
  }

  override def distinct(
      attr: Attribute,
  ): Expression = {
    val collectSetExpr = AggregateExpression(
      CollectSet(Cast(attr, StringType)),
      Complete,
      isDistinct = false
    )
    ConcatWs(Seq(Literal(distinctOperator), collectSetExpr))
  }

  override def aggregate(
      attr: Attribute,
  ): Expression = {
    val collectSetExpr = AggregateExpression(
      CollectSet(Cast(attr, StringType)),
      Complete,
      isDistinct = false
    )
    ConcatWs(Seq(Literal(aggregateOperator), collectSetExpr))
  }
}

// Stop current SparkSession so extensions are applied on a fresh one
spark.stop()

val sparkCustom = SparkSession.builder()
  .master("local[*]")
  .appName("notebook-custom-string-provenance")
  .withExtensions(
    new SparkProvenanceExtension(
      provenanceBuilder = BracketStringProvenanceBuilder
    )
  )
  .config("spark.provenance.enabled", "true")
  .getOrCreate()

println(s"Spark provenance enabled: ${sparkCustom.conf.get("spark.provenance.enabled")}")
sparkCustom.sparkContext.setLogLevel("ERROR")

cmd9.sc:1920: object creation impossible.
Missing implementations for 2 members.
  // Members declared in org.dataprov.dp.wringlet.WindowFinalizeProvenanceOperation
  def windowFinalize(attr: org.apache.spark.sql.catalyst.expressions.Attribute): org.apache.spark.sql.catalyst.expressions.Expression = ???
  
  // Members declared in org.dataprov.dp.wringlet.WindowRawProvenanceOperation
  def windowRaw(attr: org.apache.spark.sql.catalyst.expressions.Attribute): org.apache.spark.sql.catalyst.expressions.Expression = ???

object BracketStringProvenanceBuilder extends ProvenanceBuilder {
       ^
Compilation Failed

In [ ]:

val df1: DataFrame = sparkCustom.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("d", "b", "e"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C")

df1.show()

val df1WithProvenance = df1.addProvenanceColumn(col("A"))
df1WithProvenance.printSchema()
df1WithProvenance.show(false)

val df2WithProvenance : DataFrame = df1WithProvenance
    .select("A", "B")
    .join(df1WithProvenance.select("B", "C"), "B")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df2WithProvenance.show(false)

df1WithProvenance.createOrReplaceTempView("df_with_prov")
val df3WithProvenance: DataFrame = sparkCustom.sql("""
    SELECT A, B, t1.C
    FROM (
        SELECT A, C
        FROM df_with_prov
    ) AS t1
    JOIN (
        SELECT B, C
        FROM df_with_prov
    ) AS t2
    ON t1.C = t2.C
    ORDER BY A, B, t1.C
""")

df3WithProvenance.show(false)



val df4WithProvenance = df2WithProvenance.union(df3WithProvenance).distinct().orderBy("A", "B", "C")
df4WithProvenance.show(false)

val df5WithProvenance = df4WithProvenance.select("A", "C").distinct().orderBy("A", "C")
df5WithProvenance.show(false)



+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+

root
 |-- A: string (nullable = true)
 |-- B: string (nullable = true)
 |-- C: string (nullable = true)
 |-- _provenance_tag: string (nullable = true)

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |a              |
|d  |b  |e  |d              |
|f  |g  |e  |f              |
+---+---+---+---------------+

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |a&a            |
|a  |b  |e  |a&d            |
|d  |b  |c  |d&a            |
|d  |b  |e  |d&d            |
|f  |g  |e  |f&f            |
+---+---+---+---------------+

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |a&a            |
|d  |b  |e  |d&d            |
|d  |g  |e  |d&f            |
|f  |b  |e  |f&d            |
|f  |g  |e  |f&f            |
+---+---+---+--------------

df1: DataFrame = [A: string, B: string ... 1 more field]
df1WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]
df2WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]
df3WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]
df4WithProvenance: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 2 more fields]
df5WithProvenance: Dataset[org.apache.spark.sql.Row] = [A: string, C: string ... 1 more field]

In [ ]:
val df6WithProvenance = df1WithProvenance
  .groupBy("B")
  .agg(
    collect_list("A").as("A_list"),
    collect_list("C").as("C_list")
)
  .orderBy("B")

df6WithProvenance.show(false)

+---+------+------+---------------+
|B  |A_list|C_list|_provenance_tag|
+---+------+------+---------------+
|b  |[a, d]|[c, e]|a || d         |
|g  |[f]   |[e]   |f              |
+---+------+------+---------------+



df6WithProvenance: Dataset[org.apache.spark.sql.Row] = [B: string, A_list: array<string> ... 2 more fields]

# Why-provenance (witness tuples only)

In [9]:
// SparkSession.getActiveSession.foreach(_.stop())
// SparkSession.getDefaultSession.foreach(_.stop())
// try { spark.stop() } catch { case _: Throwable => () }
// try { sparkCustom.stop() } catch { case _: Throwable => () }

val sparkWhy = SparkSession.builder()
  .master("local[*]")
  .appName("notebook-why-provenance")
  .withExtensions(
    new SparkProvenanceExtension(
      provenanceBuilder = SemiWhyProvenanceBuilder
    )
  )
  .config("spark.provenance.enabled", "true")
  .getOrCreate()

println(s"Spark provenance enabled: ${sparkWhy.conf.get("spark.provenance.enabled")}")

// Set log level to ERROR to reduce verbosity
sparkWhy.sparkContext.setLogLevel("ERROR")


Spark provenance enabled: true


sparkWhy: SparkSession = org.apache.spark.sql.classic.SparkSession@6ab1b8c3

In [10]:
val dfWhy = sparkWhy.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("a", "b", "c"),
        ("d", "b", "e"),
        ("f", "g", "e"),
        ("d", "b", "e"),
        ("f", "g", null),
        ("d", null, "e"),
        (null, "g", "c")

    )
).toDF("A", "B", "C")

val dfWhyProv = dfWhy.addProvenanceColumn(col("A"))

val whyResult = dfWhyProv
  .select("A", "B")
  .join(dfWhyProv.select("B", "C"), "B")
  .select("A", "C")
  .distinct()
  .orderBy("A", "C")

whyResult.show(false)


+----+----+---------------+
|A   |C   |_provenance_tag|
+----+----+---------------+
|NULL|NULL|[f]            |
|NULL|c   |NULL           |
|NULL|e   |[f]            |
|a   |c   |[a]            |
|a   |e   |[a, d]         |
|d   |c   |[d, a]         |
|d   |e   |[d]            |
|f   |NULL|[f]            |
|f   |c   |[f]            |
|f   |e   |[f]            |
+----+----+---------------+



dfWhy: DataFrame = [A: string, B: string ... 1 more field]
dfWhyProv: DataFrame = [A: string, B: string ... 2 more fields]
whyResult: Dataset[org.apache.spark.sql.Row] = [A: string, C: string ... 1 more field]

In [11]:
val dfSales: DataFrame = sparkWhy.createDataFrame(
    Seq(
        ("a0","A", Date.valueOf("2026-01-15"), 10.0, 90),
        ("a1","A", Date.valueOf("2026-01-16"), 10.0, 120),
        ("a2","A", Date.valueOf("2026-01-17"), 5.0, 300),
        ("b0","B", Date.valueOf("2026-01-15"), 100.0, 20),
        ("b1","B", Date.valueOf("2026-01-16"), 100.0, 30),
        ("c0","C", Date.valueOf("2026-01-17"), 80.0, 60),
        ("c1","C", Date.valueOf("2026-01-18"), 82.0, 50),
        ("c2","C", Date.valueOf("2026-01-18"), 78.0, 70),
        ("d0","D", Date.valueOf("2026-01-16"), 50.0, 200),
        ("d1","D", Date.valueOf("2026-01-18"), 82.0, 50),
        ("e0","E", Date.valueOf("2026-01-15"), 82.0, 50)
    )
).toDF("id","product", "date", "price", "sales")

val dfSalesProv = dfSales.addProvenanceColumn(col("id"))
dfSalesProv.show(false)

+---+-------+----------+-----+-----+---------------+
|id |product|date      |price|sales|_provenance_tag|
+---+-------+----------+-----+-----+---------------+
|a0 |A      |2026-01-15|10.0 |90   |a0             |
|a1 |A      |2026-01-16|10.0 |120  |a1             |
|a2 |A      |2026-01-17|5.0  |300  |a2             |
|b0 |B      |2026-01-15|100.0|20   |b0             |
|b1 |B      |2026-01-16|100.0|30   |b1             |
|c0 |C      |2026-01-17|80.0 |60   |c0             |
|c1 |C      |2026-01-18|82.0 |50   |c1             |
|c2 |C      |2026-01-18|78.0 |70   |c2             |
|d0 |D      |2026-01-16|50.0 |200  |d0             |
|d1 |D      |2026-01-18|82.0 |50   |d1             |
|e0 |E      |2026-01-15|82.0 |50   |e0             |
+---+-------+----------+-----+-----+---------------+



dfSales: DataFrame = [id: string, product: string ... 3 more fields]
dfSalesProv: DataFrame = [id: string, product: string ... 4 more fields]

In [12]:
val aggreg = dfSalesProv.groupBy("product").agg(
    max("date").as("first_date"),
    min("date").as("last_date"),
    round(mean("price"), 2).as("average_price"),
    sum("sales").as("total_sales")
)
aggreg.show(false)

val aggreg2 = dfSalesProv.groupBy("product").agg(
    max("date").as("first_date")
)
aggreg2.show(false)


+-------+----------+----------+-------------+-----------+---------------+
|product|first_date|last_date |average_price|total_sales|_provenance_tag|
+-------+----------+----------+-------------+-----------+---------------+
|A      |2026-01-17|2026-01-15|8.33         |510        |[a0, a1, a2]   |
|B      |2026-01-16|2026-01-15|100.0        |50         |[b0, b1]       |
|C      |2026-01-18|2026-01-17|80.0         |180        |[c0, c1, c2]   |
|D      |2026-01-18|2026-01-16|66.0         |250        |[d0, d1]       |
|E      |2026-01-15|2026-01-15|82.0         |50         |[e0]           |
+-------+----------+----------+-------------+-----------+---------------+

+-------+----------+---------------+
|product|first_date|_provenance_tag|
+-------+----------+---------------+
|A      |2026-01-17|[a2]           |
|B      |2026-01-16|[b1]           |
|C      |2026-01-18|[c2]           |
|D      |2026-01-18|[d1]           |
|E      |2026-01-15|[e0]           |
+-------+----------+---------------+


aggreg: DataFrame = [product: string, first_date: date ... 4 more fields]
aggreg2: DataFrame = [product: string, first_date: date ... 1 more field]

In [13]:
val distinctFirstDate = aggreg.select("first_date").distinct().orderBy("first_date")
distinctFirstDate.show(false)

+----------+---------------+
|first_date|_provenance_tag|
+----------+---------------+
|2026-01-15|[e0]           |
|2026-01-16|[b0, b1]       |
|2026-01-17|[a0, a1, a2]   |
|2026-01-18|[d0, d1]       |
+----------+---------------+



distinctFirstDate: Dataset[org.apache.spark.sql.Row] = [first_date: date, _provenance_tag: array<string>]

In [14]:
val df = sparkWhy.createDataFrame(
    Seq(
        ("a0", "Ball", "2", "10.0"),
        ("b0", "Bike", "10", "500.0"),
        ("a1", "Ball", "5", "7.5"),
        ("a2", "Ball", null, "7.5"),
        ("b1", "Bike", "15", "450.0"),
        ("c0", "Car", "1", "20000.0"),
        ("c1", "Car", "2", "19500.0"),
        ("c2", "Car", "3", "19000.0"),
        ("d0", "Drone", "4", "300.0"),
        ("d1", "Drone", null, "300.0"),
        ("e0", "E-Scooter", null, null)
    )
).toDF("id", "product", "sales", "price")
val dfProv = df.addProvenanceColumn(col("id"))
dfProv.show(false)

val test = dfProv.select("product", "sales")
  .join(dfProv.select("product", "price"), "product")
  .select("product", "price")
  .distinct()
  .orderBy("product")

test.show(false)

val test2 = dfProv.groupBy("product").agg(
    round(mean("price"), 2).as("average_price")
).orderBy("product")

test2.show(false)

val test3 = dfProv.select("product").distinct().orderBy("product")

test3.show(false)


+---+---------+-----+-------+---------------+
|id |product  |sales|price  |_provenance_tag|
+---+---------+-----+-------+---------------+
|a0 |Ball     |2    |10.0   |a0             |
|b0 |Bike     |10   |500.0  |b0             |
|a1 |Ball     |5    |7.5    |a1             |
|a2 |Ball     |NULL |7.5    |a2             |
|b1 |Bike     |15   |450.0  |b1             |
|c0 |Car      |1    |20000.0|c0             |
|c1 |Car      |2    |19500.0|c1             |
|c2 |Car      |3    |19000.0|c2             |
|d0 |Drone    |4    |300.0  |d0             |
|d1 |Drone    |NULL |300.0  |d1             |
|e0 |E-Scooter|NULL |NULL   |e0             |
+---+---------+-----+-------+---------------+

+---------+-------+---------------+
|product  |price  |_provenance_tag|
+---------+-------+---------------+
|Ball     |10.0   |[a0]           |
|Ball     |7.5    |[a2]           |
|Bike     |450.0  |[b1]           |
|Bike     |500.0  |[b0]           |
|Car      |19000.0|[c2]           |
|Car      |19500.0|[c

df: DataFrame = [id: string, product: string ... 2 more fields]
dfProv: DataFrame = [id: string, product: string ... 3 more fields]
test: Dataset[org.apache.spark.sql.Row] = [product: string, price: string ... 1 more field]
test2: Dataset[org.apache.spark.sql.Row] = [product: string, average_price: double ... 1 more field]
test3: Dataset[org.apache.spark.sql.Row] = [product: string, _provenance_tag: array<string>]

In [15]:
val newdf = sparkWhy.createDataFrame(
    Seq(
        ("A", Date.valueOf("2026-01-15"), 10.0, 90),
        ("A", Date.valueOf("2026-01-16"), 10.0, 120),
        ("A", Date.valueOf("2026-01-17"), 5.0, 300),
        ("B", Date.valueOf("2026-01-15"), 100.0, 20),
        ("C", Date.valueOf("2026-01-17"), 80.0, 60),
        ("C", Date.valueOf("2026-01-18"), 82.0, 50)
    )
).toDF("product", "date", "price", "sales").addProvenanceColumn

val aggMin = newdf.groupBy("product").agg(
    //min("date").as("first_date"),
    max("price").as("max_price"),
    first("sales").as("first_sales"),
    //last("sales").as("last_sales")
    //round(mean("price"), 2).as("average_price")
    //sum("sales").as("total_sales")
)
aggMin.show(false)

+-------+---------+-----------+----------------------------------------------------------------------------+
|product|max_price|first_sales|_provenance_tag                                                             |
+-------+---------+-----------+----------------------------------------------------------------------------+
|A      |10.0     |90         |[bceecf89-9da2-4664-88e0-3e50e2934d43, 86299660-6749-4099-b473-d267ec765826]|
|B      |100.0    |20         |[f16a27f0-a33d-4880-ad9e-d0bf0fd2f37b]                                      |
|C      |82.0     |60         |[d828af06-3813-4e42-b510-007eb8976391, 9d07553b-15a5-4077-ac39-22016f9a6d42]|
+-------+---------+-----------+----------------------------------------------------------------------------+



newdf: DataFrame = [product: string, date: date ... 3 more fields]
aggMin: DataFrame = [product: string, max_price: double ... 2 more fields]